In [1]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from controlnet_aux import CannyDetector
from PIL import Image
import matplotlib.pyplot as plt



/Users/brageramberg/opt/miniconda3/envs/bannergen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/brageramberg/opt/miniconda3/envs/bannergen/lib/python3.10/site-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(


In [ ]:
# Device setup explicitly for Apple Silicon (MPS)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Global dtype for numerical stability on MPS
dtype = torch.float16

# Models (compatible and publicly accessible)
models = {
    "Stable Diffusion v1.5": "runwayml/stable-diffusion-v1-5",
    "Openjourney": "prompthero/openjourney"
}

# ControlNet setup (Canny) with global dtype
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=dtype,
    use_safetensors=True
).to(device)

# Pipeline kwargs explicitly set with global dtype
pipe_kwargs = {"torch_dtype": dtype, "use_safetensors": True, "safety_checker": None}

# Enable attention slicing for better performance on MPS
# (Will be enabled later after pipeline initialization)

# ControlNet auxiliary processor (Canny edge detector)
canny = CannyDetector()

# Input/control image
control_image = Image.open("nike.png").resize((512, 512))
control_image_canny = canny(control_image)
control_image_canny.save("canny_test_v2.png")


Using device: mps


In [ ]:
# Prompt setup
prompt = "A banner image of a summer forest, vibrant colors"

# Seeds for reproducibility
def make_generator(seed):
    return torch.Generator(device=device).manual_seed(seed)

seeds = [42, 123, 999]

# Warmup function (recommended once per session)
def warmup_pipe(pipeline, control_image):
    _ = pipeline(prompt, num_inference_steps=1, image=control_image)

# Generate and store images
results = {}
for model_name, model_path in models.items():
    print(f"\nGenerating images with model: {model_name}")
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        model_path,
        controlnet=controlnet,
        **pipe_kwargs
    ).to(device)

    pipe.enable_attention_slicing()
    warmup_pipe(pipe, control_image)

    model_images = []
    for seed in seeds:
        generator = make_generator(seed)

        image = pipe(
            prompt,
            num_inference_steps=20,
            guidance_scale=7.5,
            image=control_image,
            generator=generator
        ).images[0]

        model_images.append(image)

    results[model_name] = model_images

# Plot results
fig, axes = plt.subplots(len(models), len(seeds), figsize=(5 * len(seeds), 5 * len(models)))

for i, (model_name, images) in enumerate(results.items()):
    for j, img in enumerate(images):
        ax = axes[i, j] if len(models) > 1 else axes[j]
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"{model_name}\nSeed: {seeds[j]}")

plt.tight_layout()
plt.savefig("model_comparison_v2.png", dpi=300)
plt.show()


Generating images with model: Stable Diffusion v1.5


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 13.29it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 20/20 [00:35<00:00,  1.76s/it]



Generating images with model: Openjourney


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 16.72it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
 35%|███▌      | 7/20 [07:42<13:34, 62.68s/it] 